# 00 — Session Setup

**Run this notebook at the start of EVERY Colab session. Run cells top-to-bottom, one at a time.**

---
### Before running: verify your Colab Secrets

Click the **🔑 key icon** in the left sidebar. You should see these secrets with a **blue toggle** (Notebook access ON):
- `ANTHROPIC_API_KEY`
- `HF_TOKEN`  
- `GITHUB_PAT`
- `OPENAI_API_KEY` (optional)

If any toggle is grey (off), click it to turn it blue before continuing.

---
### ⚠️ Your branch name

Before running Cell 3 (clone repo), find the line `BRANCH_NAME = 'main'` and change it to **your branch name** (e.g. `'pair-1/smollm2-1.7b'` or `'advisor_experiments'`).

In [ ]:
# ── CELL 1: Verify GPU ────────────────────────────────────────────────────────
# Expected: Tesla T4, 15360 MiB
# If you see K80 or no GPU: Runtime → Disconnect and delete runtime → reconnect
!nvidia-smi
import torch
gpu_ok = torch.cuda.is_available()
if gpu_ok:
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'\n✅ GPU: {name}  ({vram:.0f} GB VRAM)')
    if vram < 14:
        print('⚠️  Less than 14 GB VRAM — you have a K80. Reconnect to get a T4.')
else:
    print('\n❌ No GPU detected. Go to Runtime → Change runtime type → T4 GPU.')

In [ ]:
# ── CELL 2: Mount Google Drive ────────────────────────────────────────────────
# A browser popup asks for permission — click Allow on everything.
# These are standard Google permissions required for Drive access.
from google.colab import drive
drive.mount('/content/drive')
import os
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Drive mounted at /content/drive')
else:
    print('❌ Drive mount failed — try running this cell again')

In [ ]:
# ── CELL 3: Clone or pull the GitHub repo ─────────────────────────────────────
# ⚠️  CHANGE BRANCH_NAME BEFORE RUNNING THIS CELL
#     Use your branch name, e.g. 'pair-1/smollm2-1.7b' or 'advisor_experiments'
#     Leaving it as 'main' will clone the main branch (advisor only).

import os, subprocess, sys
from google.colab import userdata

BRANCH_NAME = 'main'        # ← CHANGE THIS to your branch name
REPO_ORG    = 'Break-Through-Tech'
REPO_NAME   = 'Automation-Anywhere-1A-domain-specific-theme-labeling-via-slm-distillation'
PROJECT_DIR = '/content/project'

# Load PAT from Secrets
try:
    PAT = userdata.get('GITHUB_PAT')
    if not PAT:
        raise ValueError('GITHUB_PAT secret is empty')
    print(f'✅ GITHUB_PAT loaded ({len(PAT)} chars)')
except Exception as e:
    print(f'❌ GITHUB_PAT not loaded: {e}')
    print('   Fix: open the 🔑 Secrets panel, add GITHUB_PAT, and toggle Notebook access ON')
    raise SystemExit('Cannot clone without GITHUB_PAT')

REPO_URL = f'https://{PAT}@github.com/{REPO_ORG}/{REPO_NAME}.git'

def run_git(args, cwd=None):
    """Run a git command and print its output. Raises on failure."""
    result = subprocess.run(
        args, cwd=cwd, capture_output=True, text=True
    )
    if result.returncode != 0:
        # Strip the PAT from error messages before printing
        err = result.stderr.replace(PAT, '***')
        print(f'❌ Git error:\n{err}')
        raise RuntimeError(f'Git command failed: {" ".join(args)}')
    return result.stdout

if not os.path.exists(PROJECT_DIR) or not os.path.exists(f'{PROJECT_DIR}/.git'):
    # Remove any broken partial clone
    if os.path.exists(PROJECT_DIR):
        print('Removing broken project directory and re-cloning ...')
        subprocess.run(['rm', '-rf', PROJECT_DIR])
    print(f'Cloning branch "{BRANCH_NAME}" ...')
    run_git(['git', 'clone', '-b', BRANCH_NAME, REPO_URL, PROJECT_DIR])
    print(f'✅ Repository cloned to {PROJECT_DIR}')
else:
    print(f'Repository already cloned — pulling latest from "{BRANCH_NAME}" ...')
    run_git(['git', 'checkout', BRANCH_NAME], cwd=PROJECT_DIR)
    run_git(['git', 'pull', 'origin', BRANCH_NAME], cwd=PROJECT_DIR)
    print(f'✅ Branch "{BRANCH_NAME}" is up to date')

# Set git identity for pushing (change to your details)
subprocess.run(['git', '-C', PROJECT_DIR, 'config', 'user.email', 'student@cornell.edu'])
subprocess.run(['git', '-C', PROJECT_DIR, 'config', 'user.name',  'Student Name'])

# Verify key files exist
missing = [f for f in ['main.py', 'requirements.txt', 'requirements_colab.txt']
           if not os.path.exists(f'{PROJECT_DIR}/{f}')]
if missing:
    print(f'❌ Missing files after clone: {missing}')
    print('   The branch may be incomplete. Check your GitHub repo.')
else:
    print(f'✅ All required files present')

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── CELL 4: Install dependencies ──────────────────────────────────────────────
# Takes 3–4 minutes. Must run after Cell 3 (needs requirements files from repo).
import subprocess, sys

def pip_install(req_file):
    path = f'/content/project/{req_file}'
    if not __import__('os').path.exists(path):
        print(f'❌ {req_file} not found — did Cell 3 run successfully?')
        return False
    print(f'Installing from {req_file} ...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', path],
        capture_output=True, text=True
    )
    # Show any warnings or errors (last 5 lines)
    output = (result.stdout + result.stderr).strip()
    if output:
        for line in output.split('\n')[-5:]:
            if line.strip():
                print(f'  {line}')
    if result.returncode != 0:
        print(f'❌ pip failed for {req_file} (exit code {result.returncode})')
        return False
    print(f'✅ {req_file} installed')
    return True

ok_base  = pip_install('requirements.txt')
ok_colab = pip_install('requirements_colab.txt')

if ok_base and ok_colab:
    print('\n✅ All dependencies installed successfully')
else:
    print('\n⚠️  Some packages failed to install — check the errors above')

In [ ]:
# ── CELL 5: Mount Google Drive and set up folders ────────────────────────────
import os

DRIVE_ROOT = '/content/drive/MyDrive/slm-distillation'
dirs = [
    f'{DRIVE_ROOT}/data/raw',
    f'{DRIVE_ROOT}/data/processed',
    f'{DRIVE_ROOT}/data/checkpoints',
    f'{DRIVE_ROOT}/outputs',
    f'{DRIVE_ROOT}/hf_cache',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

# Point HuggingFace to Drive cache (models persist across sessions)
os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'

print('✅ Drive folders ready')
print(f'✅ HF model cache → {os.environ["HF_HOME"]}')

In [ ]:
# ── CELL 6: Load API keys from Colab Secrets ──────────────────────────────────
import os
from google.colab import userdata

def load_secret(name, required=True):
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f'  ✅ {name}')
            return True
        else:
            msg = 'required — add it in the 🔑 Secrets panel and toggle ON' if required else 'optional'
            print(f'  ⚠️  {name} is empty ({msg})')
            return not required
    except Exception:
        msg = '🔑 Add it and toggle Notebook access ON' if required else 'optional'
        print(f'  ❌ {name} not found — {msg}')
        return not required

print('Loading secrets:')
ok = all([
    load_secret('ANTHROPIC_API_KEY', required=True),
    load_secret('HF_TOKEN',          required=True),
    load_secret('OPENAI_API_KEY',    required=False),
])

if ok:
    print('\n✅ Required secrets loaded')
else:
    print('\n❌ Fix missing secrets before running the pipeline')

In [ ]:
# ── CELL 7: Verify everything is ready ───────────────────────────────────────
import os, torch

checks = [
    ('GPU available (T4)',     torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 14e9),
    ('Drive mounted',          os.path.exists('/content/drive/MyDrive')),
    ('Repo cloned',            os.path.exists('/content/project/main.py')),
    ('requirements installed', os.path.exists('/content/project/requirements.txt')),
    ('ANTHROPIC_API_KEY set',  'ANTHROPIC_API_KEY' in os.environ),
    ('HF_TOKEN set',           'HF_TOKEN' in os.environ),
    ('HF cache on Drive',      os.environ.get('HF_HOME','').startswith('/content/drive')),
]

all_ok = True
print('Setup verification:')
for label, ok in checks:
    icon = '✅' if ok else '❌'
    print(f'  {icon} {label}')
    if not ok:
        all_ok = False

print()
if all_ok:
    print('🚀 All checks passed! Scroll down to run the pipeline.')
else:
    print('⚠️  Fix the ❌ items above before running the pipeline.')

---
## Run the pipeline

Choose one of the cells below based on what you want to do.

In [ ]:
# ── Full training + evaluation run ───────────────────────────────────────────
# Runs everything: cluster → label → fine-tune → evaluate (~40–60 min on T4)
!python /content/project/main.py \
    --phase 1 \
    --config /content/project/configs/phase1_config.yaml \
    --device_mode colab

In [ ]:
# ── Evaluation only (model already trained) ───────────────────────────────────
# First update existing_run_dir in phase1_config.yaml, then run this.
# Or set the flag inline:
EXISTING_RUN = 'outputs/20260820_0014_SmolLM2-1.7B-Instruct_ep3'  # ← UPDATE THIS

!python /content/project/main.py \
    --phase 1 \
    --config /content/project/configs/phase1_config.yaml \
    --device_mode colab

In [ ]:
# ── Live demo mode ────────────────────────────────────────────────────────────
# Update ADAPTER_DIR with your run's adapter path, then run.
ADAPTER_DIR = '/content/drive/MyDrive/slm-distillation/outputs/YOUR_RUN_ID/models/lora_adapter'

!python /content/project/main.py \
    --phase 1 \
    --config /content/project/configs/phase1_config.yaml \
    --device_mode colab \
    --mode demo \
    --adapter_dir "$ADAPTER_DIR"

In [ ]:
# ── Create master CSV (joins all four pivot files) ────────────────────────────
RUN_DIR = '/content/drive/MyDrive/slm-distillation/outputs/YOUR_RUN_ID'  # ← UPDATE

!python /content/project/main.py \
    --phase 1 \
    --config /content/project/configs/phase1_config.yaml \
    --create_master_csv \
    --run_dir "$RUN_DIR"

In [ ]:
# ── Save code changes to GitHub ───────────────────────────────────────────────
import subprocess

COMMIT_MSG = 'describe your changes here'  # ← UPDATE THIS

for cmd in [
    ['git', 'add', '.'],
    ['git', 'commit', '-m', COMMIT_MSG],
    ['git', 'push'],
]:
    result = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/project')
    out = (result.stdout + result.stderr).strip()
    if out:
        print(out)

print('✅ Done')

---
## First session only — create your branch

**Skip this if your branch already exists on GitHub.**

Run Cell 3 first (to clone main), then run the cell below to create your branch.

In [ ]:
# ── Create your branch (run ONCE on first session) ───────────────────────────
import subprocess

MY_BRANCH = 'pair-X/model-name'   # ← CHANGE THIS (e.g. 'pair-1/smollm2-1.7b')

for cmd in [
    ['git', 'checkout', '-b', MY_BRANCH],
    ['git', 'push', '-u', 'origin', MY_BRANCH],
]:
    result = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/project')
    out = (result.stdout + result.stderr).strip()
    if out:
        print(out)

print(f"\n✅ Branch '{MY_BRANCH}' created.")
print(f"Now change BRANCH_NAME to '{MY_BRANCH}' in Cell 3 for all future sessions.")